DELIVERABLE 2 

Task 1 part 1 

Here we wrote a function that reads files ending in the '.log' suffix, built to handle both utf-8 and latin-1 encodings. 
It starts of by trying the first encoding (utf-8), and continues to the next encoding (latin-1) if the UnicodeDecodeError occurs. 

In [ ]:

from pathlib import Path 

def read_file(root_dir, pattern='*.log', encodings=['utf-8', 'latin-1']):
    root_path = Path(root_dir)
    results = []

    for filepath in root_path.rglob(pattern):
        if filepath.is_file():
            print(f'Reading file: {filepath}')

            for encoding in encodings:
                try:
                    content = filepath.read_text(encoding=encoding)
                    results.append((content, encoding))
                    break
                except UnicodeDecodeError:
                    continue 
    return results 

# Sjekker at den funksjonen funker 

# 1. Call function
all_files_data = read_file('data')

# 2. Loop through every file found in the folder
for content, used_encoding in all_files_data:
    print(f"\n--- Successfully read using: {used_encoding} ---")
    
    # 3. Split the text into lines and take the first 5 for each file
    first_few_lines = content.splitlines()[:5]
    
    # 4. Print them out
    for line in first_few_lines:
        print(line)

Reading file: data\2026\student_union\POS_01\register_sales.log
Reading file: data\2026\student_union\POS_02\register_sales.log
Reading file: data\2026\student_union\POS_03\register_sales.log
Reading file: data\2026\science_building\POS_01\register_sales.log
Reading file: data\2026\science_building\POS_02\register_sales.log
Reading file: data\2026\science_building\POS_03\register_sales.log
Reading file: data\2026\library_cafe\POS_01\register_sales.log
Reading file: data\2026\library_cafe\POS_02\register_sales.log
Reading file: data\2026\library_cafe\POS_03\register_sales.log

--- Successfully read using: utf-8 ---
[2026-10-15 07:00:26] [POS_01] item=Cappuccino price=4.05 NOK (SUCCESS)
[DEBUG] BARCODE SCANNER REBOOTED
[2026-10-15 07:03:01] [POS_01] item=Latte price=4.52 EUR (REFUND)
[2026-10-15 07:05:09] [POS_01] item=Sandwich price=7.94 USD (SUCCESS)
[2026-10-15 07:06:21] [POS_01] item=Espresso price=3.03 NOK (SUCCESS)

--- Successfully read using: utf-8 ---
[2026-10-15 07:01:49] [POS_

Task1 part 2 

Here we wrote code with regex that specifies the form/pattern of the data we wish to retrieve from our raw dataset, thereby filtering other elements that do not interest us/are pure noise. 

In [62]:
import re

#Regex pattern to match the log lines
timestamp = r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]"
register_id = r"\[(\w{6})\]"
item_id = r"item=(\w+)"
price = r"price=(\d+\.\d{2})"
currency = r"(EUR|NOK|USD)"
status = r"\((SUCCESS|REFUND|CANCELLED)\)"


pattern = rf"{timestamp} {register_id} {item_id} {price} {currency} {status}"

for content, encoding in all_files_data:
    for line in content.splitlines():
        match = re.fullmatch(pattern, line)
        
        if match:
            print(match.groups())

('2026-10-15 07:00:26', 'POS_01', 'Cappuccino', '4.05', 'NOK', 'SUCCESS')
('2026-10-15 07:03:01', 'POS_01', 'Latte', '4.52', 'EUR', 'REFUND')
('2026-10-15 07:05:09', 'POS_01', 'Sandwich', '7.94', 'USD', 'SUCCESS')
('2026-10-15 07:06:21', 'POS_01', 'Espresso', '3.03', 'NOK', 'SUCCESS')
('2026-10-15 07:07:27', 'POS_01', 'Sandwich', '7.00', 'EUR', 'SUCCESS')
('2026-10-15 07:10:17', 'POS_01', 'Cappuccino', '3.95', 'EUR', 'SUCCESS')
('2026-10-15 07:12:53', 'POS_01', 'Cappuccino', '4.60', 'EUR', 'SUCCESS')
('2026-10-15 07:15:28', 'POS_01', 'Croissant', '3.40', 'EUR', 'CANCELLED')
('2026-10-15 07:18:18', 'POS_01', 'Sandwich', '7.70', 'EUR', 'SUCCESS')
('2026-10-15 07:19:07', 'POS_01', 'Croissant', '3.98', 'NOK', 'SUCCESS')
('2026-10-15 07:21:45', 'POS_01', 'Muffin', '3.12', 'USD', 'SUCCESS')
('2026-10-15 07:24:30', 'POS_01', 'Cappuccino', '4.24', 'EUR', 'CANCELLED')
('2026-10-15 07:24:40', 'POS_01', 'Muffin', '3.77', 'NOK', 'SUCCESS')
('2026-10-15 07:25:27', 'POS_01', 'Sandwich', '7.70', 'EUR

Task 2 Part 1 

Here we wrote a function that recursively searches a directory for .log files, and returns a list of path objects. 

In [72]:
#### TASK 2 part 1

from pathlib import Path
from typing import List, Union

def find_log_files(root_dir: Union[str, Path]) -> List[Path]:
    """
    Recursively searches a directory for all .log files and returns a list of Path objects.
    """
    # 1. Convert the input path to a pathlib.Path object
    root_path = Path(root_dir)
    log_files: List[Path] = []
    
    if not root_path.exists() or not root_path.is_dir():
        return log_files

    # 2. Iterate through items using Path.iterdir()
    for item in root_path.iterdir():
        # 3. If an item is a directory, recursively search inside it
        if item.is_dir():
            log_files.extend(find_log_files(item))
            
        # 4. If an item is a file with a matching extension (.log), add its Path object
        elif item.is_file() and item.suffix.lower() == '.log':
            log_files.append(item)
            
    return log_files

In [ ]:
### Task 2 part 2 

from typing import Any, Set

def get_active_register_ids(data: Any) -> set[str]:
    """
    Recursively traverses nested dictionaries and lists to find 
    register ID strings where 'active' is true.
    """
    active_ids: set[str] = set()
    
    if isinstance(data, dict):
        # Check if the current dictionary is active and has an identifier
        if data.get("active") is True:
            reg_id = data.get("register_id") or data.get("id")
            if isinstance(reg_id, str):
                active_ids.add(reg_id)
                
        # Recursively search all values in the dictionary
        for value in data.values():
            active_ids.update(get_active_register_ids(value))
            
    elif isinstance(data, list):
        # Recursively search all items in the list
        for item in data:
            active_ids.update(get_active_register_ids(item))
            
    return active_ids

In [ ]:
### TASK 3 part 1 

import numpy as np
from typing import Union, Sequence

def remove_price_outliers(prices: Union[Sequence[float], np.ndarray]) -> np.ndarray:
    """
    Converts input price data to a 1D NumPy float array, detects hardware 
    glitch outliers (> 3 standard deviations from the mean), replaces them 
    with the median of the valid prices, and returns the cleaned array.
    """
    # 1. Convert input price data to a 1D NumPy float array (using np.array to ensure it's writable)
    arr = np.array(prices, dtype=float).ravel()
    
    if arr.size == 0:
        return arr

    # 2. Detect hardware glitch outliers (> 3 standard deviations from the mean)
    mean_val = np.mean(arr)
    std_val = np.std(arr)
    
    if std_val == 0:
        return arr  

    is_outlier = np.abs(arr - mean_val) > (3 * std_val)

    # 3. Compute the median of the valid prices
    valid_prices = arr[~is_outlier]
    if valid_prices.size > 0:
        median_valid = np.median(valid_prices)
    else:
        median_valid = np.median(arr)

    # 4. Replace detected outlier values with the median of the valid prices
    arr[is_outlier] = median_valid


    return arr



In [96]:
### TASK 3 part 2

import numpy as np

def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Computes the element-wise refund ratio using NumPy array operations,
    adding epsilon = 10^-6 to prevent zero-division errors.
    """
    epsilon = 1e-6
    return refund_count / (sales_count + epsilon)

In [97]:
#Task 4 part 1 


import pandas as pd
from typing import List, Dict, Any

def process_transactions(transactions: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Converts a list of transaction dictionaries into a pandas DataFrame,
    converts the timestamp column to datetime, sets it as the index,
    and sorts the DataFrame chronologically.
    """
    # 1. Convert the list of transaction dictionaries into a pandas.DataFrame
    df = pd.DataFrame(transactions)
    
    if df.empty or 'timestamp' not in df.columns:
        return df

    # 2. Convert the timestamp column to datetime and set it as the DataFrame index
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.set_index('timestamp', inplace=True)

    # 3. Sort the DataFrame chronologically by timestamp
    df.sort_index(inplace=True)

    return df

In [98]:
# Task 4 part 2 

import pandas as pd

def resample_item_sales(df: pd.DataFrame) -> pd.DataFrame:
    """
    Groups data by item, resamples transactions in 1-hour windows using 
    the datetime index, and calculates the total price (sum).
    """
    if df.empty or 'item' not in df.columns or not isinstance(df.index, pd.DatetimeIndex):
        return pd.DataFrame()

    # Group by 'item', resample the datetime index in 1-hour intervals, and sum the 'price'
    resampled_df = df.groupby('item').resample('1h')['price'].sum().reset_index()
    
    return resampled_df

In [102]:
#Task 4 part 3 

import pandas as pd

def create_revenue_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a pivot table displaying total transaction revenue (sum of price) 
    grouped by item across status categories (SUCCESS, REFUND, CANCELLED).
    """
    if df.empty or 'item' not in df.columns or 'status' not in df.columns or 'price' not in df.columns:
        return pd.DataFrame()

    # Define the expected status categories
    status_categories = ['SUCCESS', 'REFUND', 'CANCELLED']

    # Create the pivot table
    pivot_df = df.pivot_table(
        index='item',
        columns='status',
        values='price',
        aggfunc='sum',
        fill_value=0.0
    )

    # Ensure all status columns are present, even if missing from the data
    for status in status_categories:
        if status not in pivot_df.columns:
            pivot_df[status] = 0.0

    # Reorder columns to match the requested order
    pivot_df = pivot_df[status_categories]

    return pivot_df

In [109]:
#TASK 4 part 4 

import pandas as pd
import numpy as np


def calculate_item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the refund ratio per item by aggregating sales and refund counts 
    and calling compute_refund_ratio.
    """
    if df.empty or 'item' not in df.columns or 'status' not in df.columns:
        return pd.DataFrame()

    summary = df.groupby('item').agg(
        sales_count=('status', lambda x: (x == 'SUCCESS').sum()), # or total transactions depending on definition
        refund_count=('status', lambda x: (x == 'REFUND').sum())
    )

    # Convert to NumPy arrays for the compute_refund_ratio function
    sales_arr = summary['sales_count'].to_numpy(dtype=float)
    refund_arr = summary['refund_count'].to_numpy(dtype=float)

    # Call the function created in task 3
    ratios = compute_refund_ratio(sales_arr, refund_arr)

    # Attach the results back into a summary DataFrame
    summary['refund_ratio'] = ratios

    return summary[['sales_count', 'refund_count', 'refund_ratio']]

In [111]:
import re
import json
from pathlib import Path
from typing import Union
import numpy as np
import pandas as pd

def run_pipeline(root_dir: Union[str, Path] = 'data', nok_multiplier: float = 10.0, output_dir: Union[str, Path] = 'output') -> None:
    """
    Executes the complete end-to-end data pipeline adhering to the exact 
    input sources and output filenames required for the assignment.
    """
    root_path = Path(root_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    print("=== Step 1: Finding log files and active registers ===")
    log_files = find_log_files(root_path)
    print(f"Found {len(log_files)} log file(s).")

    # Load store metadata 
    active_registers = set()
    config_file = root_path / "store_config.json"
    
    if config_file.exists():
        try:
            config_data = json.loads(config_file.read_text(encoding='utf-8'))
            active_registers.update(get_active_register_ids(config_data))
            print(f"Loaded store metadata from: {config_file}")
        except Exception as e:
            print(f"Could not parse {config_file}: {e}")
    else:
        # Fallback search if path structure varies
        for json_file in root_path.rglob("*.json"):
            try:
                data = json.loads(json_file.read_text(encoding='utf-8'))
                active_registers.update(get_active_register_ids(data))
            except Exception as e:
                print(f"Could not parse JSON {json_file}: {e}")
                
    print(f"Identified {len(active_registers)} active register(s).")

    print("\n=== Step 2: Ingesting text logs and parsing records ===")
    raw_records = []
    
    # Regex pattern from Task 1
    timestamp_pat = r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]"
    register_pat = r"\[(\w{6})\]"
    item_pat = r"item=(\w+)"
    price_pat = r"price=(\d+\.\d{2})"
    currency_pat = r"(EUR|NOK|USD)"
    status_pat = r"\((SUCCESS|REFUND|CANCELLED)\)"
    
    line_pattern = rf"{timestamp_pat} {register_pat} {item_pat} {price_pat} {currency_pat} {status_pat}"

    for filepath in log_files:
        for encoding in ['utf-8', 'latin-1']:
            try:
                content = filepath.read_text(encoding=encoding)
                break
            except UnicodeDecodeError:
                continue
        else:
            continue

        for line in content.splitlines():
            match = re.fullmatch(line_pattern, line.strip())
            if match:
                ts, reg_id, item, price_str, currency, status = match.groups()
                
                # Filter by active registers if configuration exists
                if active_registers and reg_id not in active_registers:
                    continue

                raw_records.append({
                    'timestamp': ts,
                    'register_id': reg_id,
                    'item': item,
                    'price': float(price_str),
                    'currency': currency,
                    'status': status
                })

    print(f"Successfully parsed {len(raw_records)} transaction records.")

    if not raw_records:
        print("No records found to process. Exiting pipeline.")
        return

    print("\n=== Step 3: Building initial DataFrame ===")
    df = process_transactions(raw_records)

    print("\n=== Step 4: Applying NumPy outlier cleaning ===")
    if 'price' in df.columns:
        cleaned_prices = remove_price_outliers(df['price'].to_numpy())
        df['price_cleaned'] = cleaned_prices

    print("\n=== Step 5: Applying currency correction (NOK) ===")
    if 'currency' in df.columns:
        nok_mask = df['currency'] == 'NOK'
        df.loc[nok_mask, 'price_cleaned'] *= nok_multiplier

    print("\n=== Step 6: Generating analytical reports ===")
    # Update main price column with cleaned and corrected values
    df['price'] = df['price_cleaned']

    resampled_sales = resample_item_sales(df)
    revenue_pivot = create_revenue_pivot_table(df)
    refund_ratios_df = calculate_item_refund_ratios(df)

    print("\n=== Step 7: Saving required output files ===")
    
    # 1. cleaned_transactions.csv
    transactions_out = out_path / "cleaned_transactions.csv"
    df.to_csv(transactions_out)
    print(f"✔ Saved: {transactions_out}")

    # 2. hourly_revenue.json 
    hourly_out = out_path / "hourly_revenue.json"
    resampled_sales.to_json(hourly_out, orient='records', indent=4, date_format='iso')
    print(f"✔ Saved: {hourly_out}")

    # 3. status_pivot_table.csv
    pivot_out = out_path / "status_pivot_table.csv"
    revenue_pivot.to_csv(pivot_out)
    print(f"✔ Saved: {pivot_out}")

    # 4. item_refund_ratios.csv
    ratios_out = out_path / "item_refund_ratios.csv"
    refund_ratios_df.to_csv(ratios_out)
    print(f"✔ Saved: {ratios_out}")

    print("\nPipeline execution complete! All required output files generated successfully.")


run_pipeline('data', nok_multiplier=10.0, output_dir='output')

=== Step 1: Finding log files and active registers ===
Found 9 log file(s).
Loaded store metadata from: data\store_config.json
Identified 3 active register(s).

=== Step 2: Ingesting text logs and parsing records ===
Successfully parsed 720 transaction records.

=== Step 3: Building initial DataFrame ===

=== Step 4: Applying NumPy outlier cleaning ===

=== Step 5: Applying currency correction (NOK) ===

=== Step 6: Generating analytical reports ===

=== Step 7: Saving required output files ===
✔ Saved: output\cleaned_transactions.csv
✔ Saved: output\hourly_revenue.json
✔ Saved: output\status_pivot_table.csv
✔ Saved: output\item_refund_ratios.csv

Pipeline execution complete! All required output files generated successfully.


C:\Users\joste\AppData\Local\Temp\ipykernel_8136\2956988210.py:21: Pandas4Warning: The inplace keyword in DataFrame.set_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.set_index('timestamp', inplace=True)
C:\Users\joste\AppData\Local\Temp\ipykernel_8136\2956988210.py:24: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.sort_index(inplace=True)
